# Lesson 3: Building an Agent Reasoning Loop

## Setup

Run the first code cell before anything else: it adds the repo root to `sys.path` and loads `ANTHROPIC_API_KEY` via `helper` (including from `.env`). Then run the `nest_asyncio` cell before importing `utils`.

In [7]:
import sys
from pathlib import Path

_root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (
            (p / "helper.py").is_file()
            and (p / "utils.py").is_file()
            and (p / "agent_helpers.py").is_file()
        )
    ),
    None,
)
if _root is None:
    raise RuntimeError(
        "Could not find project root (helper.py + utils.py + agent_helpers.py). "
        "Open the training repo folder or run from a lesson subfolder inside it."
    )
sys.path.insert(0, str(_root))

from helper import get_anthropic_api_key
ANTHROPIC_API_KEY = get_anthropic_api_key()

In [15]:
import nest_asyncio
nest_asyncio.apply()

## Load the data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

## Setup the Query Tools

In [ ]:
from utils import get_doc_tools

vector_tool, summary_tool = get_doc_tools("metagpt.pdf", "metagpt")

## Setup Function Calling Agent

Current **LlamaIndex** uses **`AgentWorkflow`** + **`FunctionAgent`** instead of the removed **`FunctionCallingAgentWorker`** / **`AgentRunner`**. This notebook uses **`agent_helpers.tool_calling_workflow_from_tools`** and **`agent_helpers.run_agent_workflow_sync`** (with a shared **`ChatMemoryBuffer`** for multi-turn chat).

In [17]:
from llama_index.llms.anthropic import Anthropic

llm = Anthropic(
    model="claude-sonnet-4-6",
    temperature=0,
    api_key=ANTHROPIC_API_KEY,
)

In [18]:
from llama_index.core.memory import ChatMemoryBuffer
from agent_helpers import run_agent_workflow_sync, tool_calling_workflow_from_tools

# Agent worker is responsible for executing the next step of the agent
# Agent runner (Agent Task Orchestrator) - overall task dispatcher - creates the task, orchestrates the runs of agent works on top of a task, returns back
# the final response to the user
agent = tool_calling_workflow_from_tools([vector_tool, summary_tool], llm=llm, verbose=True)
agent_memory = ChatMemoryBuffer.from_defaults(llm=llm, token_limit=8192)

In [22]:
response = run_agent_workflow_sync(
    agent,
    "Tell me about the agent roles in MetaGPT, "
    "and then how they communicate with each other.",
    memory=agent_memory,
)

[tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the agent roles in MetaGPT, and then...', chat_history=None, memory=ChatMemoryBuffer(chat_store=SimpleChatStore(store={'chat_history': [ChatMessage(r...
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[4 items], current_agent_name='Agent')
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[4 items], current_agent_name='Agent')
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with no result


AsyncLibraryNotFoundError: unknown async library, or not in async context

In [ ]:
_vn = vector_tool.query_engine.query(
    "What are the agent roles in MetaGPT and how do they communicate?"
)
print(_vn.source_nodes[0].get_content(metadata_mode="all"))

In [ ]:
response = run_agent_workflow_sync(
    agent,
    "Tell me about the evaluation datasets used.",
    memory=agent_memory,
)

In [ ]:
response = run_agent_workflow_sync(
    agent,
    "Tell me the results over one of the above datasets.",
    memory=agent_memory,
)

## Lower-Level: Debuggability and Control

Older **AgentRunner** APIs (`create_task`, `run_step`, …) were removed in current **LlamaIndex**. Multi-step tool use happens inside **`AgentWorkflow.run`**. Use **`run_agent_workflow_sync`** with a shared **`agent_memory`** for each turn.

For finer-grained inspection, run `handler = agent.run(...)` and `await handler.stream_events()` from an **async** context.

In [ ]:
# Another turn on the same conversation memory
dbg = run_agent_workflow_sync(
    agent,
    "What about how agents share information?",
    memory=agent_memory,
)
print(dbg)
